# HanuAi ML Assessment — Task 2: Advanced EDA and Text Mining
Dataset: `Hanuai_task2_assignment.csv`. This notebook checks data quality, summarizes key fields, extracts issue tags from free text, and applies TF-IDF + K-Means for exploratory clustering. Tags and clusters are not verified engineering root causes.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, ast, re
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
df = pd.read_csv("Hanuai_task2_assignment.csv")
print("Shape:", df.shape); display(df.head())


In [ ]:
# Data quality: types, nulls, exact duplicates, and event identifier uniqueness
display(pd.DataFrame({"dtype":df.dtypes.astype(str),"missing_count":df.isna().sum(),"missing_pct":(df.isna().mean()*100).round(2)}))
print("Exact duplicate rows:",df.duplicated().sum())
print("Duplicate Event id occurrences:",df["Event id"].duplicated().sum())


In [ ]:
# Parse dates without silently deleting invalid records
for col in ["Opened date","BUILD_DATE","IN_USE_DATE"]:
    df[col+"_parsed"] = pd.to_datetime(df[col],errors="coerce")
df["Opened Year"] = df["Opened date_parsed"].dt.year
text_cols=["CAUSAL_VERBATIM","CORRECTION_VERBATIM","CUSTOMER_VERBATIM"]
for col in text_cols: df[col]=df[col].fillna("").astype(str)
display(df.describe(include="all").T)
for col in ["CAUSAL_CD_DESC","COMPLAINT_CD_DESC","MAKE","MODEL","BUILD_PLANT_DESC","MODLYR"]:
    display(df[col].value_counts(dropna=False).head(10))
df["Opened Year"].value_counts().sort_index().plot(kind="bar",title="Events by opened year"); plt.tight_layout(); plt.show()


In [ ]:
# Transparent keyword rules; a record may receive multiple tags.
def extract_tags(row):
    text=" ".join(str(row[c]) for c in text_cols).lower()
    patterns={
      "Radio/audio unit":r"\bradio\b|\baudio\b|amplifier",
      "Display/black screen":r"\bdisplay\b|\bscreen\b|black screen|blank screen",
      "Connectivity/network":r"\bwifi\b|\bwi-fi\b|connect(?:ion|ing)?|ethernet|onstar|hotspot|network",
      "Electrical/power":r"\bpower\b|voltage|ground circuit|battery|electrical",
      "Communication/data bus":r"lost communication|data bus|ethernet|\bdtc\b|module communication",
      "Controls/buttons":r"button|knob|control",
      "Software/programming":r"software|program(?:ming|med)?|update|sps|reset",
      "Sound/no audio":r"no sound|no audio|sound cuts|audio cuts",
      "Intermittent operation":r"intermittent|cuts out|quit working|comes back",
      "Replacement/repair":r"replac(?:e|ed|ement)|install(?:ed|ation)?"
    }
    tags=[tag for tag,pattern in patterns.items() if re.search(pattern,text)]
    return tags or ["Other/unspecified"]
df["Issue Tags List"]=df.apply(extract_tags,axis=1)
df["Issue Tags"]=df["Issue Tags List"].apply(lambda x:"; ".join(x))
df["Primary Issue Category"]=df["Issue Tags List"].str[0]
tag_counts=Counter(tag for tags in df["Issue Tags List"] for tag in tags)
tag_summary=pd.DataFrame(tag_counts.most_common(),columns=["Extracted Tag","Event Count"])
display(tag_summary)
tag_summary.head(10).sort_values("Event Count").plot(kind="barh",x="Extracted Tag",y="Event Count",legend=False); plt.tight_layout(); plt.show()


In [ ]:
# Exploratory topic groups from combined narratives using TF-IDF and K-Means
text=(df["CUSTOMER_VERBATIM"]+" "+df["CAUSAL_VERBATIM"]+" "+df["CORRECTION_VERBATIM"]).str.lower()
vectorizer=TfidfVectorizer(stop_words="english",max_features=1500,ngram_range=(1,2),min_df=2)
X=vectorizer.fit_transform(text)
model=KMeans(n_clusters=5,random_state=42,n_init=10)
df["Text Cluster"]=model.fit_predict(X)+1
terms=np.array(vectorizer.get_feature_names_out())
for cluster in sorted(df["Text Cluster"].unique()):
    subset=df[df["Text Cluster"]==cluster]
    mean_tfidf=np.asarray(X[subset.index].mean(axis=0)).ravel()
    print("Cluster",cluster,"events:",len(subset),"top terms:",", ".join(terms[mean_tfidf.argsort()[-8:][::-1]]))
    display(subset[["Event id","CUSTOMER_VERBATIM","CAUSAL_CD_DESC","Issue Tags"]].head(3))


In [ ]:
# Export enriched event-level and tag-summary datasets
df.drop(columns=["Issue Tags List"],errors="ignore").to_csv("Task2_Enriched_Events.csv",index=False)
tag_summary.to_csv("Task2_Extracted_Tags_Summary.csv",index=False)
print("Export complete.")
